In [ ]:
import json
import psycopg2
from dateutil import parser as dateparser
import os
import re
import nltk
from typing import Dict, List, Generator
import time
from datetime import datetime

In [ ]:
# Ensure NLTK punkt is available
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

In [ ]:
# ---------------------------------------------------------
# DATABASE CONNECTION
# ---------------------------------------------------------
def get_db_connection():
    """Establish database connection"""
    return psycopg2.connect(
        host="ep-fancy-dream-a1a6pcdi-pooler.ap-southeast-1.aws.neon.tech",
        database="mates",
        user="neondb_owner",
        password="npg_X0N3vFLwAfEe"
    )

In [ ]:
# ---------------------------------------------------------
# TEXT PREPROCESSING FUNCTIONS
# ---------------------------------------------------------

# Category definitions for Khmer text
CATEGORIES: Dict[str, List[str]] = {
    'Agriculture': ['កសិកម្ម', 'ស្រូវ', 'ដំណាំ', 'កសិករ', 'ដីស្រែ', 'ជី', 'សត្វចិញ្ចឹម', 'ទឹកស្រោចស្រព', 'ទីផ្សារកសិផល', 'គ្រាប់ពូជ'],
    'Public': ['សាធារណៈ', 'រដ្ឋ', 'សេវាសាធារណៈ', 'អាជ្ញាធរ', 'សង្គម'],
    'Commerce': ['ពាណិជ្ជកម្ម', 'អាជីវកម្ម', 'ទីផ្សារ', 'ពាណិជ្ជកម្មអន្តរជាតិ', 'ការនាំចេញ', 'ការនាំចូល'],
    'Religion': ['សាសនា', 'ព្រះ', 'វត្ត', 'សាសនាចក្រ', 'សមាគមសាសនា'],
    'Culture': ['វប្បធម៌', 'ប្រពៃណី', 'សិល្បៈ', 'តន្ត្រី', 'រាំ', 'ភាពយន្ត', 'បុរាណ', 'ពិធីបុណ្យ'],
    'Economy': ['សេដ្ឋកិច្ច', 'ធនាគារ', 'វិនិយោគ', 'ទីផ្សារ', 'ការងារ', 'ប្រាក់ខែ', 'អតិផរណា'],
    'Education': ['អប់រំ', 'សាលា', 'សិក្សា', 'គ្រូ', 'និស្សិត', 'មហាវិទ្យាល័យ', 'សាកលវិទ្យាល័យ'],
    'Sports': ['កីឡា', 'បាល់ទាត់', 'វាយកូនបាល់', 'អត្តពលិក', 'ការប្រកួត', 'អូឡាំពិក', 'កីឡាជាតិ'],
    'Environment': ['បរិស្ថាន', 'ធម្មជាតិ', 'ទន្លេ', 'ព្រៃឈើ', 'អាកាសធាតុ', 'ការបំពុល', 'សត្វព្រៃ'],
    'International_news': ['ពត៌មានអន្តរជាតិ', 'អន្តរជាតិ', 'ពិភពលោក', 'កិច្ចប្រជុំអន្តរជាតិ', 'អង្គការសហប្រជាជាតិ'],
    'Health': ['សុខាភិបាល', 'ពេទ្យ', 'មន្ទីរពេទ្យ', 'ជំងឺ', 'ថ្នាំ', 'វេជ្ជសាស្ត្រ', 'វ៉ាក់សាំង', 'អនាម័យ'],
    'Industry': ['ឧស្សាហកម្ម', 'រោងចក្រ', 'ផលិតកម្ម', 'ឧស្សាហកម្មផលិតផល', 'សេវាឧស្សាហកម្ម'],
    'Technology': ['បច្ចេកវិទ្យា', 'កុំព្យូទ័រ', 'អ៊ិនធឺណេត', 'ឌីជីថល', 'កម្មវិធី', 'ទិន្នន័យ', 'ហេដ្ឋារចនាសម្ព័ន្ធ'],
    'National News': ['ពត៌មានជាតិ', 'សារព័ត៌មានជាតិ', 'ប្រទេស', 'រាជរដ្ឋាភិបាល', 'សភា', 'អាជ្ញាធរ'],
    'Justice': ['យុត្តិធម៍', 'ច្បាប់', 'សាលា', 'ប៉ូលិស', 'អង្គការយុត្តិធម៍'],
    'Labor': ['ការងារ', 'បុគ្គលិក', 'ប្រាក់ខែ', 'សិទ្ធិកម្មករ', 'សហភាព'],
    'Conflict War': ['សង្រ្គាម', 'ជម្លោះ', 'ការប៉ះទង្គិច', 'កងទ័ព', 'អាវុធ', 'ការវាយប្រហារ'],
    'Telecommunication': ['ទូរគមនាគមន៍', 'ទូរស័ព្ទ', 'អ៊ិនធឺណេត', 'សញ្ញា', 'ខ្សែទូរស័ព្ទ'],
    'Traffic Accident': ['គ្រោះថ្នាក់ចរាចរណ៍', 'បុកគ្នា', 'ឡានបុក', 'ម៉ូតូបុក', 'អ្នករបួស', 'ស្លាប់'],
    'Tourism': ['ទេសចរណ៍', 'អង្គរវត្ត', 'សៀមរាប', 'ឆ្នេរ', 'សណ្ឋាគារ', 'មគ្គុទ្ទេសក៍', 'ប្រាសាទ'],
    'Aviation': ['អាកាសចរណ៍', 'អាកាសយាន', 'យន្តហោះ', 'កំពង់ផែ', 'ហោះឆ្នេរ']
}

def clean_khmer_text(text: str) -> str:
    """Clean Khmer text by removing unwanted characters"""
    if not text:
        return ""
    # Keep Khmer letters, Khmer numbers, Latin numbers, Khmer punctuation
    text = re.sub(r"[^\u1780-\u17FF0-9\u17E0-\u17E9\s។៕៖,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_khmer_words(text: str) -> List[str]:
    """Tokenize Khmer text into words"""
    rough_tokens = text.split()
    tokens = []
    for token in rough_tokens:
        if len(token) > 20:
            tokens.extend(list(token))  # fallback for long glued words
        else:
            tokens.append(token)
    return tokens

def count_sentences_khmer(text: str) -> int:
    """Count sentences in Khmer text"""
    sentences = re.split(r"[។៕]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return len(sentences)

def extract_tags_and_category(title: str, content: str) -> tuple:
    """Extract tags and determine category from text"""
    text = title + " " + content
    tags = set()
    category_count = {cat: 0 for cat in CATEGORIES}

    for cat, keywords in CATEGORIES.items():
        for kw in keywords:
            if kw in text:
                tags.add(kw)
                category_count[cat] += 1

    # Determine primary category
    primary_category = None
    category_confidence = 0.0
    if category_count:
        sorted_cats = sorted(category_count.items(), key=lambda x: x[1], reverse=True)
        if sorted_cats[0][1] > 0:
            primary_category = sorted_cats[0][0]
            total_matches = sum(category_count.values())
            category_confidence = sorted_cats[0][1] / total_matches if total_matches > 0 else 0.0

    return list(tags), primary_category, category_confidence

def preprocess_article(article_data: dict) -> dict:
    """Preprocess a single article"""
    title = clean_khmer_text(article_data.get("title", ""))
    content = clean_khmer_text(article_data.get("content", ""))

    tags, primary_category, category_confidence = extract_tags_and_category(title, content)

    words = tokenize_khmer_words(content)
    word_count = len(words)
    sentence_count = count_sentences_khmer(content)
    character_count = len(content.replace(" ", ""))

    return {
        "url": article_data.get("url", ""),
        "source": article_data.get("source", ""),
        "publication_date": article_data.get("publication_date", ""),
        "scrape_date": article_data.get("scrape_date", ""),
        "title": title,
        "content": content,
        "tags": tags,
        "word_count": word_count,
        "sentence_count": sentence_count,
        "character_count": character_count,
        "primary_category": primary_category,
        "category_confidence": category_confidence
    }



In [ ]:

# ---------------------------------------------------------
# DATABASE SCHEMA CREATION
# ---------------------------------------------------------
def create_normalized_tables(conn):
    """Create all normalized tables"""
    create_tables_sql = """
    -- Drop tables if they exist (for clean setup)
    DROP TABLE IF EXISTS article_tags CASCADE;
    DROP TABLE IF EXISTS articles CASCADE;
    DROP TABLE IF EXISTS tags CASCADE;
    DROP TABLE IF EXISTS sources CASCADE;
    DROP TABLE IF EXISTS categories CASCADE;

    -- Create categories table
    CREATE TABLE categories (
        category_id SERIAL PRIMARY KEY,
        category_name VARCHAR(100) UNIQUE NOT NULL,
        description TEXT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Create sources table
    CREATE TABLE sources (
        source_id SERIAL PRIMARY KEY,
        source_url VARCHAR(500) UNIQUE NOT NULL,
        source_name VARCHAR(200) NOT NULL,
        is_active BOOLEAN DEFAULT TRUE,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Create tags table
    CREATE TABLE tags (
        tag_id SERIAL PRIMARY KEY,
        tag_name VARCHAR(100) UNIQUE NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Create articles table
    CREATE TABLE articles (
        article_id SERIAL PRIMARY KEY,
        url VARCHAR(1000) UNIQUE NOT NULL,
        source_id INTEGER REFERENCES sources(source_id),
        publication_date DATE NOT NULL,
        scrape_date TIMESTAMP NOT NULL,
        title TEXT NOT NULL,
        content TEXT,
        word_count INTEGER DEFAULT 0,
        sentence_count INTEGER DEFAULT 0,
        character_count INTEGER DEFAULT 0,
        category_id INTEGER REFERENCES categories(category_id),
        category_confidence DECIMAL(3,2) DEFAULT 0.0,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    );

    -- Create article_tags junction table
    CREATE TABLE article_tags (
        article_tag_id SERIAL PRIMARY KEY,
        article_id INTEGER REFERENCES articles(article_id) ON DELETE CASCADE,
        tag_id INTEGER REFERENCES tags(tag_id) ON DELETE CASCADE,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        UNIQUE(article_id, tag_id)
    );

    -- Create indexes for better performance
    CREATE INDEX idx_articles_url ON articles(url);
    CREATE INDEX idx_articles_publication_date ON articles(publication_date);
    CREATE INDEX idx_articles_source_id ON articles(source_id);
    CREATE INDEX idx_articles_category_id ON articles(category_id);
    CREATE INDEX idx_articles_created_at ON articles(created_at);
    CREATE INDEX idx_article_tags_article_id ON article_tags(article_id);
    CREATE INDEX idx_article_tags_tag_id ON article_tags(tag_id);
    CREATE INDEX idx_sources_url ON sources(source_url);
    CREATE INDEX idx_categories_name ON categories(category_name);
    CREATE INDEX idx_tags_name ON tags(tag_name);
    """
    
    try:
        cur = conn.cursor()
        cur.execute(create_tables_sql)
        conn.commit()
        cur.close()
        print("Normalized tables created successfully!")
        return True
    except Exception as e:
        print(f"Table creation failed: {e}")
        return False



In [ ]:

# ---------------------------------------------------------
# DATABASE HELPER FUNCTIONS
# ---------------------------------------------------------
def get_or_create_source(conn, source_url):
    """Get or create source record"""
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO sources (source_url, source_name)
            VALUES (%s, %s)
            ON CONFLICT (source_url) DO UPDATE SET source_name = EXCLUDED.source_name
            RETURNING source_id;
        """, (source_url, source_url))
        return cur.fetchone()[0]
    finally:
        cur.close()

def get_or_create_category(conn, category_name):
    """Get or create category record"""
    if not category_name:
        return None

    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO categories (category_name)
            VALUES (%s)
            ON CONFLICT (category_name) DO UPDATE SET category_name = EXCLUDED.category_name
            RETURNING category_id;
        """, (category_name,))
        return cur.fetchone()[0]
    except Exception as e:
        print(f"Error creating category {category_name}: {e}")
        return None
    finally:
        cur.close()

def get_or_create_tag(conn, tag_name):
    """Get or create tag record"""
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO tags (tag_name)
            VALUES (%s)
            ON CONFLICT (tag_name) DO UPDATE SET tag_name = EXCLUDED.tag_name
            RETURNING tag_id;
        """, (tag_name,))
        return cur.fetchone()[0]
    finally:
        cur.close()

def insert_article(conn, article_data, source_id, category_id):
    """Insert article into database"""
    try:
        publication_date = dateparser.parse(article_data["publication_date"]).date()
        scrape_date = dateparser.parse(article_data["scrape_date"])
    except Exception as e:
        print(f"Date parsing error for article {article_data.get('url', 'unknown')}: {e}")
        # Use current date as fallback
        publication_date = datetime.now().date()
        scrape_date = datetime.now()

    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO articles (
                url, source_id, category_id,
                publication_date, scrape_date,
                title, content,
                word_count, sentence_count, character_count,
                category_confidence
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO UPDATE SET
                title = EXCLUDED.title,
                content = EXCLUDED.content,
                word_count = EXCLUDED.word_count,
                sentence_count = EXCLUDED.sentence_count,
                character_count = EXCLUDED.character_count,
                category_id = EXCLUDED.category_id,
                category_confidence = EXCLUDED.category_confidence
            RETURNING article_id;
        """, (
            article_data["url"],
            source_id,
            category_id,
            publication_date,
            scrape_date,
            article_data["title"],
            article_data["content"],
            article_data.get("word_count", 0),
            article_data.get("sentence_count", 0),
            article_data.get("character_count", 0),
            article_data.get("category_confidence", 0.0)
        ))
        
        article_id = cur.fetchone()[0]
        conn.commit()
        return article_id
    except Exception as e:
        conn.rollback()
        print(f"Error inserting article {article_data.get('url', 'unknown')}: {e}")
        return None
    finally:
        cur.close()

def insert_article_tags(conn, article_id, tags):
    """Insert tags for an article"""
    if not tags:
        return
    
    cur = conn.cursor()
    try:
        for tag_name in tags:
            tag_id = get_or_create_tag(conn, tag_name)
            cur.execute("""
                INSERT INTO article_tags (article_id, tag_id)
                VALUES (%s, %s)
                ON CONFLICT (article_id, tag_id) DO NOTHING;
            """, (article_id, tag_id))
        conn.commit()
    except Exception as e:
        conn.rollback()
        print(f"Error inserting tags for article {article_id}: {e}")
    finally:
        cur.close()


In [ ]:

# ---------------------------------------------------------
# BATCH PROCESSING FUNCTIONS
# ---------------------------------------------------------
def batch_generator(data, batch_size=1000):
    """Generate batches from data"""
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

def process_batch(conn, batch_articles):
    """Process a batch of articles"""
    processed_count = 0
    error_count = 0
    
    for article in batch_articles:
        try:
            # 1. Preprocess article
            preprocessed_article = preprocess_article(article)
            
            # 2. Insert / get source
            source_id = get_or_create_source(conn, preprocessed_article["source"])
            
            # 3. Insert / get category
            category_id = get_or_create_category(conn, preprocessed_article.get("primary_category"))
            
            # 4. Insert article
            article_id = insert_article(conn, preprocessed_article, source_id, category_id)
            
            if article_id:
                # 5. Insert tags
                tags = preprocessed_article.get("tags", [])
                insert_article_tags(conn, article_id, tags)
                
                processed_count += 1
                if processed_count % 100 == 0:
                    print(f"  Processed {processed_count} articles in current batch...")
            else:
                error_count += 1
                
        except Exception as e:
            error_count += 1
            print(f"Error processing article {article.get('url', 'unknown')}: {e}")
            continue
    
    return processed_count, error_count

def run_complete_etl(json_path, batch_size=1000, reset_tables=False):
    """
    Run complete ETL pipeline with preprocessing and batch processing
    
    Args:
        json_path: Path to input JSON file
        batch_size: Number of articles to process per batch (default: 1000)
        reset_tables: Whether to recreate tables (default: False)
    """
    print("=" * 80)
    print("STARTING COMPLETE ETL PIPELINE")
    print("=" * 80)
    
    # Load JSON data
    print(f"Loading JSON from {json_path}...")
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if isinstance(data, dict):
            data = [data]
        
        print(f"Loaded {len(data)} articles")
    except Exception as e:
        print(f"Error loading JSON: {e}")
        return
    
    # Setup database connection
    print("Connecting to database...")
    conn = get_db_connection()
    conn.autocommit = False  # We'll manage transactions manually for batches
    
    # Create tables if requested
    if reset_tables:
        print("Creating normalized tables...")
        create_normalized_tables(conn)
    
    total_processed = 0
    total_errors = 0
    start_time = time.time()
    
    # Process in batches
    print(f"\nProcessing in batches of {batch_size} articles...")
    for batch_num, batch in enumerate(batch_generator(data, batch_size), 1):
        batch_start_time = time.time()
        print(f"\n{'='*60}")
        print(f"PROCESSING BATCH {batch_num} ({len(batch)} articles)")
        print(f"{'='*60}")
        
        processed, errors = process_batch(conn, batch)
        
        batch_time = time.time() - batch_start_time
        total_processed += processed
        total_errors += errors
        
        print(f"Batch {batch_num} completed:")
        print(f"  - Successfully processed: {processed}")
        print(f"  - Errors: {errors}")
        print(f"  - Time: {batch_time:.2f} seconds")
        print(f"  - Avg time per article: {batch_time/max(len(batch), 1):.2f} seconds")
        
        # Commit after each batch
        try:
            conn.commit()
        except Exception as e:
            print(f"Error committing batch {batch_num}: {e}")
            conn.rollback()
    
    # Cleanup
    conn.close()
    
    total_time = time.time() - start_time
    print(f"\n{'='*80}")
    print("ETL PIPELINE COMPLETED")
    print(f"{'='*80}")
    print(f"Total articles processed: {total_processed}")
    print(f"Total errors: {total_errors}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average time per article: {total_time/max(total_processed, 1):.2f} seconds")
    print(f"Success rate: {(total_processed/(total_processed + total_errors)*100):.1f}%")


In [ ]:
# ---------------------------------------------------------
# MAIN EXECUTION
# ---------------------------------------------------------
if __name__ == "__main__":
    # Configuration
    INPUT_JSON = r"D:\Menghour\MATES\data\datasets\raw\all_articles_cleaned.json"
    BATCH_SIZE = 1000  # Process 1000 articles per batch
    RESET_TABLES = True  # Set to False if you don't want to drop existing tables
    
    # Run the complete ETL pipeline
    run_complete_etl(
        json_path=INPUT_JSON,
        batch_size=BATCH_SIZE,
        reset_tables=RESET_TABLES
    )

In [ ]:
# %%
import json
import psycopg2
from dateutil import parser as dateparser
import os
import re
import nltk
from typing import Dict, List, Tuple
import time
from datetime import datetime
import hashlib
import shutil
from pathlib import Path
import traceback

# Ensure NLTK punkt is available
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

# %%
# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
class Config:
    # Database configuration
    DB_HOST = "ep-fancy-dream-a1a6pcdi-pooler.ap-southeast-1.aws.neon.tech"
    DB_NAME = "mates"
    DB_USER = "neondb_owner"
    DB_PASSWORD = "npg_X0N3vFLwAfEe"
    
    # File paths
    RAW_DATA_DIR = r"D:\Menghour\MATES\data\datasets\raw"
    PROCESSED_DATA_DIR = r"D:\Menghour\MATES\data\datasets\processed"
    ARCHIVE_DATA_DIR = r"D:\Menghour\MATES\data\datasets\archive"
    LOG_FILE = r"D:\Menghour\MATES\data\etl_log.json"
    
    # Processing settings
    BATCH_SIZE = 1000
    MAX_RETRIES = 3
    RETRY_DELAY = 5  # seconds

# %%
# ---------------------------------------------------------
# LOGGER CLASS FOR TRACKING PROCESSED FILES
# ---------------------------------------------------------
class ETLLogger:
    def __init__(self, log_file: str):
        self.log_file = log_file
        self.log_data = self._load_log()
    
    def _load_log(self) -> dict:
        """Load log data from file"""
        if os.path.exists(self.log_file):
            try:
                with open(self.log_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except:
                return {"processed_files": [], "failed_files": [], "stats": {}}
        return {"processed_files": [], "failed_files": [], "stats": {}}
    
    def save_log(self):
        """Save log data to file"""
        os.makedirs(os.path.dirname(self.log_file), exist_ok=True)
        with open(self.log_file, 'w', encoding='utf-8') as f:
            json.dump(self.log_data, f, ensure_ascii=False, indent=2)
    
    def is_file_processed(self, file_path: str) -> bool:
        """Check if file has been processed"""
        file_hash = self._get_file_hash(file_path)
        for processed in self.log_data["processed_files"]:
            if processed.get("file_path") == file_path and processed.get("file_hash") == file_hash:
                return True
        return False
    
    def mark_file_processed(self, file_path: str, processed_count: int, failed_count: int):
        """Mark file as processed"""
        file_hash = self._get_file_hash(file_path)
        self.log_data["processed_files"].append({
            "file_path": file_path,
            "file_hash": file_hash,
            "processed_date": datetime.now().isoformat(),
            "processed_count": processed_count,
            "failed_count": failed_count
        })
        self.save_log()
    
    def mark_file_failed(self, file_path: str, error: str):
        """Mark file as failed"""
        file_hash = self._get_file_hash(file_path)
        self.log_data["failed_files"].append({
            "file_path": file_path,
            "file_hash": file_hash,
            "failed_date": datetime.now().isoformat(),
            "error": error
        })
        self.save_log()
    
    def _get_file_hash(self, file_path: str) -> str:
        """Generate MD5 hash of file"""
        try:
            with open(file_path, 'rb') as f:
                return hashlib.md5(f.read()).hexdigest()
        except:
            return "error"
    
    def get_stats(self) -> dict:
        """Get processing statistics"""
        if "stats" not in self.log_data:
            self.log_data["stats"] = {}
        return self.log_data["stats"]
    
    def update_stats(self, key: str, value):
        """Update statistics"""
        if "stats" not in self.log_data:
            self.log_data["stats"] = {}
        self.log_data["stats"][key] = value
        self.save_log()

# %%
# ---------------------------------------------------------
# DATABASE MANAGER CLASS
# ---------------------------------------------------------
class DatabaseManager:
    def __init__(self, config: Config):
        self.config = config
        self.connection = None
    
    def connect(self):
        """Establish database connection"""
        self.connection = psycopg2.connect(
            host=self.config.DB_HOST,
            database=self.config.DB_NAME,
            user=self.config.DB_USER,
            password=self.config.DB_PASSWORD
        )
        self.connection.autocommit = False
        return self.connection
    
    def close(self):
        """Close database connection"""
        if self.connection:
            self.connection.close()
            self.connection = None
    
    def create_tables_if_not_exist(self):
        """Create normalized tables if they don't exist"""
        create_tables_sql = """
        -- Create categories table if not exists
        CREATE TABLE IF NOT EXISTS categories (
            category_id SERIAL PRIMARY KEY,
            category_name VARCHAR(100) UNIQUE NOT NULL,
            description TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        -- Create sources table if not exists
        CREATE TABLE IF NOT EXISTS sources (
            source_id SERIAL PRIMARY KEY,
            source_url VARCHAR(500) UNIQUE NOT NULL,
            source_name VARCHAR(200) NOT NULL,
            is_active BOOLEAN DEFAULT TRUE,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        -- Create tags table if not exists
        CREATE TABLE IF NOT EXISTS tags (
            tag_id SERIAL PRIMARY KEY,
            tag_name VARCHAR(100) UNIQUE NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        -- Create articles table if not exists
        CREATE TABLE IF NOT EXISTS articles (
            article_id SERIAL PRIMARY KEY,
            url VARCHAR(1000) UNIQUE NOT NULL,
            source_id INTEGER REFERENCES sources(source_id),
            publication_date DATE NOT NULL,
            scrape_date TIMESTAMP NOT NULL,
            title TEXT NOT NULL,
            content TEXT,
            word_count INTEGER DEFAULT 0,
            sentence_count INTEGER DEFAULT 0,
            character_count INTEGER DEFAULT 0,
            category_id INTEGER REFERENCES categories(category_id),
            category_confidence DECIMAL(3,2) DEFAULT 0.0,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        );

        -- Create article_tags junction table if not exists
        CREATE TABLE IF NOT EXISTS article_tags (
            article_tag_id SERIAL PRIMARY KEY,
            article_id INTEGER REFERENCES articles(article_id) ON DELETE CASCADE,
            tag_id INTEGER REFERENCES tags(tag_id) ON DELETE CASCADE,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE(article_id, tag_id)
        );

        -- Create processing_log table for tracking
        CREATE TABLE IF NOT EXISTS processing_log (
            log_id SERIAL PRIMARY KEY,
            file_name VARCHAR(500),
            file_hash VARCHAR(64),
            processed_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            article_count INTEGER,
            status VARCHAR(20),
            error_message TEXT
        );

        -- Create indexes for better performance
        CREATE INDEX IF NOT EXISTS idx_articles_url ON articles(url);
        CREATE INDEX IF NOT EXISTS idx_articles_publication_date ON articles(publication_date);
        CREATE INDEX IF NOT EXISTS idx_articles_source_id ON articles(source_id);
        CREATE INDEX IF NOT EXISTS idx_articles_category_id ON articles(category_id);
        CREATE INDEX IF NOT EXISTS idx_articles_created_at ON articles(created_at);
        CREATE INDEX IF NOT EXISTS idx_article_tags_article_id ON article_tags(article_id);
        CREATE INDEX IF NOT EXISTS idx_article_tags_tag_id ON article_tags(tag_id);
        CREATE INDEX IF NOT EXISTS idx_sources_url ON sources(source_url);
        CREATE INDEX IF NOT EXISTS idx_categories_name ON categories(category_name);
        CREATE INDEX IF NOT EXISTS idx_tags_name ON tags(tag_name);
        """
        
        try:
            cur = self.connection.cursor()
            cur.execute(create_tables_sql)
            self.connection.commit()
            cur.close()
            print("✓ Database tables verified/created")
            return True
        except Exception as e:
            print(f"✗ Table creation failed: {e}")
            self.connection.rollback()
            return False
    
    def log_processing(self, file_name: str, file_hash: str, article_count: int, 
                      status: str = "success", error_message: str = None):
        """Log processing activity to database"""
        try:
            cur = self.connection.cursor()
            cur.execute("""
                INSERT INTO processing_log (file_name, file_hash, article_count, status, error_message)
                VALUES (%s, %s, %s, %s, %s)
            """, (file_name, file_hash, article_count, status, error_message))
            self.connection.commit()
            cur.close()
        except Exception as e:
            print(f"Warning: Failed to log to database: {e}")
            self.connection.rollback()
    
    def is_file_processed_in_db(self, file_name: str, file_hash: str) -> bool:
        """Check if file is already processed in database"""
        try:
            cur = self.connection.cursor()
            cur.execute("""
                SELECT COUNT(*) FROM processing_log 
                WHERE file_name = %s AND file_hash = %s AND status = 'success'
            """, (file_name, file_hash))
            count = cur.fetchone()[0]
            cur.close()
            return count > 0
        except:
            return False

# %%
# ---------------------------------------------------------
# TEXT PREPROCESSING FUNCTIONS
# ---------------------------------------------------------

# Category definitions for Khmer text
CATEGORIES: Dict[str, List[str]] = {
    'Agriculture': ['កសិកម្ម', 'ស្រូវ', 'ដំណាំ', 'កសិករ', 'ដីស្រែ', 'ជី', 'សត្វចិញ្ចឹម', 'ទឹកស្រោចស្រព', 'ទីផ្សារកសិផល', 'គ្រាប់ពូជ'],
    'Public': ['សាធារណៈ', 'រដ្ឋ', 'សេវាសាធារណៈ', 'អាជ្ញាធរ', 'សង្គម'],
    'Commerce': ['ពាណិជ្ជកម្ម', 'អាជីវកម្ម', 'ទីផ្សារ', 'ពាណិជ្ជកម្មអន្តរជាតិ', 'ការនាំចេញ', 'ការនាំចូល'],
    'Religion': ['សាសនា', 'ព្រះ', 'វត្ត', 'សាសនាចក្រ', 'សមាគមសាសនា'],
    'Culture': ['វប្បធម៌', 'ប្រពៃណី', 'សិល្បៈ', 'តន្ត្រី', 'រាំ', 'ភាពយន្ត', 'បុរាណ', 'ពិធីបុណ្យ'],
    'Economy': ['សេដ្ឋកិច្ច', 'ធនាគារ', 'វិនិយោគ', 'ទីផ្សារ', 'ការងារ', 'ប្រាក់ខែ', 'អតិផរណា'],
    'Education': ['អប់រំ', 'សាលា', 'សិក្សា', 'គ្រូ', 'និស្សិត', 'មហាវិទ្យាល័យ', 'សាកលវិទ្យាល័យ'],
    'Sports': ['កីឡា', 'បាល់ទាត់', 'វាយកូនបាល់', 'អត្តពលិក', 'ការប្រកួត', 'អូឡាំពិក', 'កីឡាជាតិ'],
    'Environment': ['បរិស្ថាន', 'ធម្មជាតិ', 'ទន្លេ', 'ព្រៃឈើ', 'អាកាសធាតុ', 'ការបំពុល', 'សត្វព្រៃ'],
    'International_news': ['ពត៌មានអន្តរជាតិ', 'អន្តរជាតិ', 'ពិភពលោក', 'កិច្ចប្រជុំអន្តរជាតិ', 'អង្គការសហប្រជាជាតិ'],
    'Health': ['សុខាភិបាល', 'ពេទ្យ', 'មន្ទីរពេទ្យ', 'ជំងឺ', 'ថ្នាំ', 'វេជ្ជសាស្ត្រ', 'វ៉ាក់សាំង', 'អនាម័យ'],
    'Industry': ['ឧស្សាហកម្ម', 'រោងចក្រ', 'ផលិតកម្ម', 'ឧស្សាហកម្មផលិតផល', 'សេវាឧស្សាហកម្ម'],
    'Technology': ['បច្ចេកវិទ្យា', 'កុំព្យូទ័រ', 'អ៊ិនធឺណេត', 'ឌីជីថល', 'កម្មវិធី', 'ទិន្នន័យ', 'ហេដ្ឋារចនាសម្ព័ន្ធ'],
    'National News': ['ពត៌មានជាតិ', 'សារព័ត៌មានជាតិ', 'ប្រទេស', 'រាជរដ្ឋាភិបាល', 'សភា', 'អាជ្ញាធរ'],
    'Justice': ['យុត្តិធម៍', 'ច្បាប់', 'សាលា', 'ប៉ូលិស', 'អង្គការយុត្តិធម៍'],
    'Labor': ['ការងារ', 'បុគ្គលិក', 'ប្រាក់ខែ', 'សិទ្ធិកម្មករ', 'សហភាព'],
    'Conflict War': ['សង្រ្គាម', 'ជម្លោះ', 'ការប៉ះទង្គិច', 'កងទ័ព', 'អាវុធ', 'ការវាយប្រហារ'],
    'Telecommunication': ['ទូរគមនាគមន៍', 'ទូរស័ព្ទ', 'អ៊ិនធឺណេត', 'សញ្ញា', 'ខ្សែទូរស័ព្ទ'],
    'Traffic Accident': ['គ្រោះថ្នាក់ចរាចរណ៍', 'បុកគ្នា', 'ឡានបុក', 'ម៉ូតូបុក', 'អ្នករបួស', 'ស្លាប់'],
    'Tourism': ['ទេសចរណ៍', 'អង្គរវត្ត', 'សៀមរាប', 'ឆ្នេរ', 'សណ្ឋាគារ', 'មគ្គុទ្ទេសក៍', 'ប្រាសាទ'],
    'Aviation': ['អាកាសចរណ៍', 'អាកាសយាន', 'យន្តហោះ', 'កំពង់ផែ', 'ហោះឆ្នេរ']
}

def clean_khmer_text(text: str) -> str:
    """Clean Khmer text by removing unwanted characters"""
    if not text:
        return ""
    # Keep Khmer letters, Khmer numbers, Latin numbers, Khmer punctuation
    text = re.sub(r"[^\u1780-\u17FF0-9\u17E0-\u17E9\s។៕៖,]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_khmer_words(text: str) -> List[str]:
    """Tokenize Khmer text into words"""
    rough_tokens = text.split()
    tokens = []
    for token in rough_tokens:
        if len(token) > 20:
            tokens.extend(list(token))  # fallback for long glued words
        else:
            tokens.append(token)
    return tokens

def count_sentences_khmer(text: str) -> int:
    """Count sentences in Khmer text"""
    sentences = re.split(r"[។៕]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return len(sentences)

def extract_tags_and_category(title: str, content: str) -> tuple:
    """Extract tags and determine category from text"""
    text = title + " " + content
    tags = set()
    category_count = {cat: 0 for cat in CATEGORIES}

    for cat, keywords in CATEGORIES.items():
        for kw in keywords:
            if kw in text:
                tags.add(kw)
                category_count[cat] += 1

    # Determine primary category
    primary_category = None
    category_confidence = 0.0
    if category_count:
        sorted_cats = sorted(category_count.items(), key=lambda x: x[1], reverse=True)
        if sorted_cats[0][1] > 0:
            primary_category = sorted_cats[0][0]
            total_matches = sum(category_count.values())
            category_confidence = sorted_cats[0][1] / total_matches if total_matches > 0 else 0.0

    return list(tags), primary_category, category_confidence

def preprocess_article(article_data: dict) -> dict:
    """Preprocess a single article"""
    title = clean_khmer_text(article_data.get("title", ""))
    content = clean_khmer_text(article_data.get("content", ""))

    tags, primary_category, category_confidence = extract_tags_and_category(title, content)

    words = tokenize_khmer_words(content)
    word_count = len(words)
    sentence_count = count_sentences_khmer(content)
    character_count = len(content.replace(" ", ""))

    return {
        "url": article_data.get("url", ""),
        "source": article_data.get("source", ""),
        "publication_date": article_data.get("publication_date", ""),
        "scrape_date": article_data.get("scrape_date", ""),
        "title": title,
        "content": content,
        "tags": tags,
        "word_count": word_count,
        "sentence_count": sentence_count,
        "character_count": character_count,
        "primary_category": primary_category,
        "category_confidence": category_confidence
    }

# %%
# ---------------------------------------------------------
# FILE PROCESSING FUNCTIONS
# ---------------------------------------------------------
def split_into_batches(input_file: str, output_dir: str, batch_size: int = 1000) -> List[str]:
    """
    Split a large JSON file into smaller batch files
    Returns list of created batch files
    """
    os.makedirs(output_dir, exist_ok=True)
    batch_files = []
    
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        if not isinstance(data, list):
            data = [data]
        
        total_articles = len(data)
        print(f"Splitting {total_articles} articles into batches of {batch_size}...")
        
        for i in range(0, total_articles, batch_size):
            batch_num = i // batch_size + 1
            batch_data = data[i:i + batch_size]
            
            batch_filename = f"batch_{batch_num:03d}_{len(batch_data)}_articles.json"
            batch_filepath = os.path.join(output_dir, batch_filename)
            
            with open(batch_filepath, 'w', encoding='utf-8') as f:
                json.dump(batch_data, f, ensure_ascii=False, indent=2)
            
            batch_files.append(batch_filepath)
            print(f"  Created {batch_filename} with {len(batch_data)} articles")
        
        print(f"✓ Created {len(batch_files)} batch files")
        return batch_files
        
    except Exception as e:
        print(f"✗ Error splitting file: {e}")
        return []

def preprocess_batch_file(input_file: str, output_file: str) -> Tuple[int, int]:
    """
    Preprocess a batch file
    Returns: (processed_count, error_count)
    """
    try:
        with open(input_file, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        if not isinstance(data, list):
            data = [data]
        
        processed_articles = []
        error_count = 0
        
        for article in data:
            try:
                processed_article = preprocess_article(article)
                processed_articles.append(processed_article)
            except Exception as e:
                error_count += 1
                print(f"  Warning: Failed to preprocess article: {e}")
                continue
        
        # Save preprocessed data
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump(processed_articles, f, ensure_ascii=False, indent=2)
        
        return len(processed_articles), error_count
        
    except Exception as e:
        print(f"✗ Error preprocessing batch file: {e}")
        return 0, 1

def archive_file(source_file: str, archive_dir: str):
    """Move file to archive directory"""
    try:
        os.makedirs(archive_dir, exist_ok=True)
        filename = os.path.basename(source_file)
        archive_path = os.path.join(archive_dir, filename)
        
        # If file already exists in archive, add timestamp
        if os.path.exists(archive_path):
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            name, ext = os.path.splitext(filename)
            archive_path = os.path.join(archive_dir, f"{name}_{timestamp}{ext}")
        
        shutil.move(source_file, archive_path)
        print(f"  Archived to: {archive_path}")
    except Exception as e:
        print(f"  Warning: Could not archive file: {e}")

# %%
# ---------------------------------------------------------
# DATABASE HELPER FUNCTIONS
# ---------------------------------------------------------
def get_or_create_source(conn, source_url: str) -> int:
    """Get or create source record"""
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO sources (source_url, source_name)
            VALUES (%s, %s)
            ON CONFLICT (source_url) DO UPDATE SET source_name = EXCLUDED.source_name
            RETURNING source_id;
        """, (source_url, source_url))
        result = cur.fetchone()
        return result[0] if result else None
    except Exception as e:
        print(f"Error creating source {source_url}: {e}")
        return None
    finally:
        cur.close()

def get_or_create_category(conn, category_name: str) -> int:
    """Get or create category record"""
    if not category_name:
        return None

    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO categories (category_name)
            VALUES (%s)
            ON CONFLICT (category_name) DO UPDATE SET category_name = EXCLUDED.category_name
            RETURNING category_id;
        """, (category_name,))
        result = cur.fetchone()
        return result[0] if result else None
    except Exception as e:
        print(f"Error creating category {category_name}: {e}")
        return None
    finally:
        cur.close()

def get_or_create_tag(conn, tag_name: str) -> int:
    """Get or create tag record"""
    cur = conn.cursor()
    try:
        cur.execute("""
            INSERT INTO tags (tag_name)
            VALUES (%s)
            ON CONFLICT (tag_name) DO UPDATE SET tag_name = EXCLUDED.tag_name
            RETURNING tag_id;
        """, (tag_name,))
        result = cur.fetchone()
        return result[0] if result else None
    except Exception as e:
        print(f"Error creating tag {tag_name}: {e}")
        return None
    finally:
        cur.close()

def load_preprocessed_to_database(conn, preprocessed_file: str, db_logger: DatabaseManager) -> Tuple[int, int]:
    """
    Load preprocessed articles to database
    Returns: (processed_count, error_count)
    """
    try:
        with open(preprocessed_file, 'r', encoding='utf-8') as f:
            articles = json.load(f)
        
        if not isinstance(articles, list):
            articles = [articles]
        
        processed_count = 0
        error_count = 0
        
        for article in articles:
            try:
                # Parse dates
                publication_date = dateparser.parse(article["publication_date"]).date()
                scrape_date = dateparser.parse(article["scrape_date"])
                
                # Get or create source
                source_id = get_or_create_source(conn, article["source"])
                if not source_id:
                    error_count += 1
                    continue
                
                # Get or create category
                category_id = get_or_create_category(conn, article.get("primary_category"))
                
                # Insert article
                cur = conn.cursor()
                cur.execute("""
                    INSERT INTO articles (
                        url, source_id, category_id,
                        publication_date, scrape_date,
                        title, content,
                        word_count, sentence_count, character_count,
                        category_confidence
                    )
                    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                    ON CONFLICT (url) DO NOTHING
                    RETURNING article_id;
                """, (
                    article["url"],
                    source_id,
                    category_id,
                    publication_date,
                    scrape_date,
                    article["title"],
                    article["content"],
                    article.get("word_count", 0),
                    article.get("sentence_count", 0),
                    article.get("character_count", 0),
                    article.get("category_confidence", 0.0)
                ))
                
                result = cur.fetchone()
                if result:
                    article_id = result[0]
                    
                    # Insert tags
                    tags = article.get("tags", [])
                    for tag_name in tags:
                        tag_id = get_or_create_tag(conn, tag_name)
                        if tag_id:
                            cur.execute("""
                                INSERT INTO article_tags (article_id, tag_id)
                                VALUES (%s, %s)
                                ON CONFLICT (article_id, tag_id) DO NOTHING;
                            """, (article_id, tag_id))
                    
                    processed_count += 1
                else:
                    # Article already exists (URL conflict)
                    pass
                
                cur.close()
                
            except Exception as e:
                error_count += 1
                print(f"  Database error for article: {e}")
                continue
        
        conn.commit()
        return processed_count, error_count
        
    except Exception as e:
        conn.rollback()
        print(f"✗ Database loading error: {e}")
        return 0, 1

# %%
# ---------------------------------------------------------
# MAIN ETL PIPELINE
# ---------------------------------------------------------
class ETLEngine:
    def __init__(self, config: Config):
        self.config = config
        self.logger = ETLLogger(config.LOG_FILE)
        self.db_manager = DatabaseManager(config)
        
        # Create directories
        for directory in [config.RAW_DATA_DIR, config.PROCESSED_DATA_DIR, config.ARCHIVE_DATA_DIR]:
            os.makedirs(directory, exist_ok=True)
    
    def find_raw_json_files(self) -> List[str]:
        """Find all JSON files in raw directory"""
        json_files = []
        for file in os.listdir(self.config.RAW_DATA_DIR):
            if file.endswith('.json'):
                json_files.append(os.path.join(self.config.RAW_DATA_DIR, file))
        
        print(f"Found {len(json_files)} JSON files in raw directory")
        return sorted(json_files)
    
    def process_single_file(self, raw_file: str) -> bool:
        """
        Process a single raw JSON file
        Returns: True if successful, False otherwise
        """
        print(f"\n{'='*80}")
        print(f"PROCESSING FILE: {os.path.basename(raw_file)}")
        print(f"{'='*80}")
        
        start_time = time.time()
        
        # Check if already processed
        file_hash = self.logger._get_file_hash(raw_file)
        if self.logger.is_file_processed(raw_file):
            print(f"✓ File already processed, skipping...")
            return True
        
        # Connect to database and check there too
        conn = self.db_manager.connect()
        if self.db_manager.is_file_processed_in_db(os.path.basename(raw_file), file_hash):
            print(f"✓ File already processed in database, skipping...")
            self.db_manager.close()
            return True
        
        # Create database tables if needed
        self.db_manager.create_tables_if_not_exist()
        
        try:
            # Step 1: Split into batches
            print(f"\n1. Splitting into batches...")
            batch_files = split_into_batches(
                raw_file, 
                self.config.RAW_DATA_DIR, 
                self.config.BATCH_SIZE
            )
            
            if not batch_files:
                print("✗ Failed to split file")
                self.logger.mark_file_failed(raw_file, "Failed to split into batches")
                return False
            
            total_processed = 0
            total_errors = 0
            
            # Step 2: Process each batch
            for batch_file in batch_files:
                batch_name = os.path.basename(batch_file)
                print(f"\n2. Processing batch: {batch_name}")
                
                # Generate output filename
                base_name = os.path.splitext(batch_name)[0]
                preprocessed_file = os.path.join(
                    self.config.PROCESSED_DATA_DIR, 
                    f"{base_name}_preprocessed.json"
                )
                
                # Preprocess batch
                print(f"   Preprocessing...")
                preprocessed_count, preprocess_errors = preprocess_batch_file(
                    batch_file, 
                    preprocessed_file
                )
                
                if preprocessed_count == 0:
                    print(f"   ✗ Preprocessing failed, skipping batch")
                    total_errors += 1
                    continue
                
                print(f"   ✓ Preprocessed {preprocessed_count} articles")
                
                # Load to database
                print(f"   Loading to database...")
                db_processed, db_errors = load_preprocessed_to_database(
                    conn, 
                    preprocessed_file, 
                    self.db_manager
                )
                
                print(f"   ✓ Loaded {db_processed} articles to database")
                
                total_processed += db_processed
                total_errors += (preprocess_errors + db_errors)
                
                # Archive batch file
                print(f"   Archiving...")
                archive_file(batch_file, self.config.ARCHIVE_DATA_DIR)
                archive_file(preprocessed_file, self.config.ARCHIVE_DATA_DIR)
            
            # Step 3: Log processing
            elapsed_time = time.time() - start_time
            
            if total_processed > 0:
                self.logger.mark_file_processed(raw_file, total_processed, total_errors)
                self.db_manager.log_processing(
                    os.path.basename(raw_file),
                    file_hash,
                    total_processed,
                    "success"
                )
                
                print(f"\n{'='*80}")
                print(f"✓ FILE PROCESSING COMPLETED")
                print(f"{'='*80}")
                print(f"Total articles processed: {total_processed}")
                print(f"Total errors: {total_errors}")
                print(f"Time taken: {elapsed_time:.2f} seconds")
                print(f"Average: {elapsed_time/max(total_processed, 1):.3f} sec/article")
                
                # Archive original file
                archive_file(raw_file, self.config.ARCHIVE_DATA_DIR)
                
                return True
            else:
                print(f"\n✗ No articles were processed successfully")
                self.logger.mark_file_failed(raw_file, "No articles processed")
                self.db_manager.log_processing(
                    os.path.basename(raw_file),
                    file_hash,
                    0,
                    "failed",
                    "No articles processed"
                )
                return False
                
        except Exception as e:
            error_msg = str(e)
            print(f"\n✗ ERROR: {error_msg}")
            traceback.print_exc()
            
            self.logger.mark_file_failed(raw_file, error_msg)
            self.db_manager.log_processing(
                os.path.basename(raw_file),
                file_hash,
                0,
                "failed",
                error_msg
            )
            return False
        
        finally:
            self.db_manager.close()
    
    def run_pipeline(self):
        """Run complete ETL pipeline on all files"""
        print(f"{'='*80}")
        print("STARTING COMPLETE ETL PIPELINE")
        print(f"{'='*80}")
        
        raw_files = self.find_raw_json_files()
        
        if not raw_files:
            print("No raw JSON files found to process!")
            return
        
        successful_files = 0
        failed_files = 0
        total_processed = 0
        total_errors = 0
        start_time = time.time()
        
        for raw_file in raw_files:
            if self.process_single_file(raw_file):
                successful_files += 1
                
                # Get stats from logger
                for processed in self.logger.log_data["processed_files"]:
                    if processed["file_path"] == raw_file:
                        total_processed += processed.get("processed_count", 0)
                        total_errors += processed.get("failed_count", 0)
            else:
                failed_files += 1
        
        total_time = time.time() - start_time
        
        print(f"\n{'='*80}")
        print("ETL PIPELINE SUMMARY")
        print(f"{'='*80}")
        print(f"Total files processed: {len(raw_files)}")
        print(f"  Successful: {successful_files}")
        print(f"  Failed:     {failed_files}")
        print(f"Total articles loaded to database: {total_processed}")
        print(f"Total errors encountered: {total_errors}")
        print(f"Total processing time: {total_time:.2f} seconds")
        print(f"Success rate: {(successful_files/len(raw_files)*100):.1f}%")
        
        # Save final stats
        self.logger.update_stats("last_run", {
            "timestamp": datetime.now().isoformat(),
            "total_files": len(raw_files),
            "successful